## Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from torch import from_numpy, Tensor
from torchmetrics import MetricCollection
import torchmetrics

In [2]:
base = Path(".").resolve() / "eval-all" / "data"
assert base.is_dir() and base.exists()
base


PosixPath('/workspaces/ts-qa/eval-all/data')

In [3]:
results = pd.read_hdf(base / "all_joined.h5", "data")
results

,setting,model,mode,sample_id,question_id,prediction_text,action_sequence,textual_description,question_type,question,answer_type,answer_text,answer,options,options_sequence,prediction_parsed_binary,prediction_parsed_multi
0,4.2.1,Falcon (7B),100,24000,0,2,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The person is starting the sequence with playi...,interval_whole_sequence_multi,What is the total count of diverse actions per...,multi,The correct answer is: 4.,2.0,"{'A': '1', 'B': '2', 'C': '4'}","[1, 2, 4]",NaN,B
1,4.2.1,Falcon (7B),100,24000,1,holding a baby,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The person is starting the sequence with playi...,count_number_multi,Can you determine the activity that the person...,multi,The correct answer is holding a baby.,0.0,"{'A': 'holding a baby', 'B': 'skipping rope', ...","[holding a baby, skipping rope, eating with th...",NaN,A
2,4.2.1,Falcon (7B),100,24000,2,playing guitar,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The person is starting the sequence with playi...,before_multi,What actions were undertaken by the person som...,multi,The correct answer is playing guitar.,2.0,"{'A': 'throwing a ball', 'B': 'T-posing', 'C':...","[throwing a ball, T-posing, playing guitar]",NaN,C
3,4.2.1,Falcon (7B),100,24000,3,correct,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The person is starting the sequence with playi...,comparison_timestamp_same_binary,Is there no notable distinction between the ac...,binary,No,0.0,NaN,NaN,True,NaN
4,4.2.1,Falcon (7B),100,24000,4,1,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The person is starting the sequence with playi...,count_open,What is the total number of times the person p...,open,1,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2684995,4.2.2,UMT5 Small,full,29999,0,correct,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The sequence starts with picking something up ...,extremum_least_binary,"In terms of occurrence, was the person's invol...",binary,True,1.0,NaN,NaN,True,NaN
2684996,4.2.2,UMT5 Small,full,29999,1,correct,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The sequence starts with picking something up ...,comparison_first_last_same_binary,Are the initial and final actions identical to...,binary,No,0.0,NaN,NaN,True,NaN
2684997,4.2.2,UMT5 Small,full,29999,2,jumping once,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The sequence starts with picking something up ...,before_multi,Some time before the person performed playing ...,multi,The correct answer is picking something up wit...,0.0,"{'A': 'picking something up with both hands', ...","[picking something up with both hands, jumping...",NaN,B
2684998,4.2.2,UMT5 Small,full,29999,3,correct,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The sequence starts with picking something up ...,extremum_most_binary,Was kicking a ball the most frequently observe...,binary,This is not correct.,0.0,NaN,NaN,True,NaN


## Binary Tasks

In [4]:
binary = results[results["answer_type"] == "binary"].copy()
binary.drop(
    columns=["options", "options_sequence", "prediction_parsed_multi"], inplace=True
)
binary["prediction_parsed_binary"].fillna(np.nan, inplace=True)
binary = binary.astype({"answer": int, "prediction_parsed_binary": float}, copy=False)
binary["prediction_parsed_binary"].replace(np.nan, pd.NA)
binary

,setting,model,mode,sample_id,question_id,prediction_text,action_sequence,textual_description,question_type,question,answer_type,answer_text,answer,prediction_parsed_binary
3,4.2.1,Falcon (7B),100,24000,3,correct,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The person is starting the sequence with playi...,comparison_timestamp_same_binary,Is there no notable distinction between the ac...,binary,No,0,1.0
6,4.2.1,Falcon (7B),100,24001,1,false,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The first activity the person is doing is kick...,first_binary,Was drinking with the left hand the first acti...,binary,This is not correct.,0,0.0
8,4.2.1,Falcon (7B),100,24001,3,correct,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The first activity the person is doing is kick...,comparison_timestamp_same_binary,Is there no notable distinction between the ac...,binary,False,0,1.0
9,4.2.1,Falcon (7B),100,24001,4,correct,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The first activity the person is doing is kick...,interval_part_sequence_binary,Is it accurate to state that the person is eng...,binary,False,0,1.0
10,4.2.1,Falcon (7B),100,24002,0,false,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The person initiates their actions with eating...,comparison_first_last_different_binary,Are the initial and final actions not the same?,binary,Not true,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2684990,4.2.2,UMT5 Small,full,29998,0,correct,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...","First, the person is bowing from 0:0.0 to 0:4....",comparison_first_last_same_binary,Are the initial and final actions identical?,binary,No,0,1.0
2684993,4.2.2,UMT5 Small,full,29998,3,false,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...","First, the person is bowing from 0:0.0 to 0:4....",comparison_timestamp_same_binary,Is the action at 1.913 equal to the action at ...,binary,This is correct!,1,0.0
2684995,4.2.2,UMT5 Small,full,29999,0,correct,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The sequence starts with picking something up ...,extremum_least_binary,"In terms of occurrence, was the person's invol...",binary,True,1,1.0
2684996,4.2.2,UMT5 Small,full,29999,1,correct,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The sequence starts with picking something up ...,comparison_first_last_same_binary,Are the initial and final actions identical to...,binary,No,0,1.0


In [6]:
def unwrap(res: dict[str, Tensor]) -> dict[str, float]:
    return {k: v.item() for k, v in res.items()}


def compute_binary_metrics(df: pd.DataFrame) -> dict[str, float]:
    metrics = MetricCollection(
        {
            "Accuracy": torchmetrics.Accuracy(task="binary"),
            "Precision": torchmetrics.Precision(task="binary"),
            "Recall": torchmetrics.Recall(task="binary"),
            "F1": torchmetrics.F1Score(task="binary"),
        }
    )

    return unwrap(
        metrics(
            from_numpy(
                df["prediction_parsed_binary"].to_numpy(na_value=-100)
            ),  # Or coin toss?
            from_numpy(df["answer"].to_numpy()),
        )
    )


all_results = (
    binary.groupby(["setting", "model", "mode"])
    .apply(compute_binary_metrics)
    .apply(pd.Series)
    .reset_index()
)
all_results

,setting,model,mode,Accuracy,F1,Precision,Recall
0,4.2.1,Falcon (7B),100,0.586047,0.584581,0.613928,0.557911
1,4.2.1,Falcon (7B),150,0.548516,0.519214,0.584615,0.466974
2,4.2.1,Falcon (7B),200,0.598692,0.624311,0.595328,0.656260
3,4.2.1,Falcon (7B),250,0.554865,0.375974,0.653386,0.263920
4,4.2.1,Falcon (7B),300,0.537931,0.326397,0.682975,0.214439
...,...,...,...,...,...,...,...
174,4.2.2,MVP,full,0.683729,0.662360,0.723770,0.610557
175,4.2.2,T5 Base,full,0.643625,0.609078,0.712639,0.531797
176,4.2.2,T5 Small,full,0.617642,0.575196,0.684769,0.495853
177,4.2.2,UMT5 Base,full,0.565196,0.601265,0.576749,0.627957


## Multi Tasks

In [7]:
multi = results[results["answer_type"] == "multi"].copy()
multi.drop(columns=["prediction_parsed_binary"], inplace=True)
multi

,setting,model,mode,sample_id,question_id,prediction_text,action_sequence,textual_description,question_type,question,answer_type,answer_text,answer,options,options_sequence,prediction_parsed_multi
0,4.2.1,Falcon (7B),100,24000,0,2,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The person is starting the sequence with playi...,interval_whole_sequence_multi,What is the total count of diverse actions per...,multi,The correct answer is: 4.,2.0,"{'A': '1', 'B': '2', 'C': '4'}","[1, 2, 4]",B
1,4.2.1,Falcon (7B),100,24000,1,holding a baby,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The person is starting the sequence with playi...,count_number_multi,Can you determine the activity that the person...,multi,The correct answer is holding a baby.,0.0,"{'A': 'holding a baby', 'B': 'skipping rope', ...","[holding a baby, skipping rope, eating with th...",A
2,4.2.1,Falcon (7B),100,24000,2,playing guitar,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The person is starting the sequence with playi...,before_multi,What actions were undertaken by the person som...,multi,The correct answer is playing guitar.,2.0,"{'A': 'throwing a ball', 'B': 'T-posing', 'C':...","[throwing a ball, T-posing, playing guitar]",C
7,4.2.1,Falcon (7B),100,24001,2,drinking with the left hand,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The first activity the person is doing is kick...,last_multi,Regarding the final action undertaken by the p...,multi,The correct answer is: drinking with the left ...,0.0,"{'A': 'drinking with the left hand', 'B': 'cat...","[drinking with the left hand, catching a ball,...",A
19,4.2.1,Falcon (7B),100,24003,4,picking something up with both hands,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The sequence starts with picking something up ...,first_multi,What was the action performed by the person as...,multi,The correct answer is: picking something up wi...,0.0,"{'A': 'picking something up with both hands', ...","[picking something up with both hands, catchin...",A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2684989,4.2.2,UMT5 Small,full,29997,4,kicking a ball,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The first activity the person is doing is play...,before_multi,What actions took place some time prior to som...,multi,The correct answer is kicking a ball.,2.0,"{'A': 'drinking with the left hand', 'B': 'thr...","[drinking with the left hand, throwing a ball,...",C
2684992,4.2.2,UMT5 Small,full,29998,2,3,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...","First, the person is bowing from 0:0.0 to 0:4....",interval_part_sequence_multi,What is the total number of activities in whic...,multi,The correct answer is 2.,1.0,"{'A': '4', 'B': '2', 'C': '3'}","[4, 2, 3]",C
2684994,4.2.2,UMT5 Small,full,29998,4,catching a ball,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...","First, the person is bowing from 0:0.0 to 0:4....",after_multi,"Following some time after bowing, what actions...",multi,The correct answer is: catching a ball.,0.0,"{'A': 'catching a ball', 'B': 'golfing (swingi...","[catching a ball, golfing (swinging a club), s...",A
2684997,4.2.2,UMT5 Small,full,29999,2,jumping once,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The sequence starts with picking something up ...,before_multi,Some time before the person performed playing ...,multi,The correct answer is picking something up wit...,0.0,"{'A': 'picking something up with both hands', ...","[picking something up with both hands, jumping...",B


In [8]:
def compute_multi_metrics(df: pd.DataFrame) -> dict[str, float]:
    metrics = MetricCollection(
        {
            "Accuracy": torchmetrics.Accuracy(task="multiclass", num_classes=3),
            "Precision": torchmetrics.Precision(task="multiclass", num_classes=3),
            "Recall": torchmetrics.Recall(task="multiclass", num_classes=3),
            "F1": torchmetrics.F1Score(task="multiclass", num_classes=3),
        }
    )

    return unwrap(
        metrics(
            from_numpy(
                df["prediction_parsed_multi"]
                .map({"A": 0, "B": 1, "C": 2})
                .to_numpy(na_value=-1)
            ),
            from_numpy(df["answer"].to_numpy()),
        )
    )


all_results_multi = (
    multi.groupby(["setting", "model", "mode"])
    .apply(compute_multi_metrics)
    .apply(pd.Series)
    .reset_index()
)
all_results_multi

,setting,model,mode,Accuracy,F1,Precision,Recall
0,4.2.1,Falcon (7B),100,0.496070,0.496070,0.496070,0.496070
1,4.2.1,Falcon (7B),150,0.495146,0.495146,0.495146,0.495146
2,4.2.1,Falcon (7B),200,0.596411,0.596411,0.596411,0.596411
3,4.2.1,Falcon (7B),250,0.676484,0.676484,0.676484,0.676484
4,4.2.1,Falcon (7B),300,0.526815,0.526815,0.526815,0.526815
...,...,...,...,...,...,...,...
174,4.2.2,MVP,full,0.416705,0.416705,0.416705,0.416705
175,4.2.2,T5 Base,full,0.403144,0.403144,0.403144,0.403144
176,4.2.2,T5 Small,full,0.363847,0.363847,0.363847,0.363847
177,4.2.2,UMT5 Base,full,0.355756,0.355756,0.355756,0.355756


## Open Tasks

In [9]:
open = results[results["answer_type"] == "open"].copy()
open.drop(
    columns=[
        "prediction_parsed_binary",
        "options",
        "answer",
        "options_sequence",
        "prediction_parsed_multi",
    ],
    inplace=True,
)
open

,setting,model,mode,sample_id,question_id,prediction_text,action_sequence,textual_description,question_type,question,answer_type,answer_text
4,4.2.1,Falcon (7B),100,24000,4,1,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The person is starting the sequence with playi...,count_open,What is the total number of times the person p...,open,1
5,4.2.1,Falcon (7B),100,24001,0,drinking with the left hand,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The first activity the person is doing is kick...,right_after_open,Which action happened right after kicking a ball?,open,The person is drinking with the left hand.
12,4.2.1,Falcon (7B),100,24002,2,"eating with the right hand, drinking with the ...","{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The person initiates their actions with eating...,count_number_open,In what activity is the person involved in 2 t...,open,eating with the right hand
13,4.2.1,Falcon (7B),100,24002,3,eating with the right hand,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The person initiates their actions with eating...,extremum_least_open,"Among the activities, which one was performed ...",open,drinking with the left hand
14,4.2.1,Falcon (7B),100,24002,4,1,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The person initiates their actions with eating...,count_open,How many occurrences of eating with the right ...,open,2
...,...,...,...,...,...,...,...,...,...,...,...,...
2684978,4.2.2,UMT5 Small,full,29995,3,correct,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The sequence starts with throwing a ball. This...,extremum_least_open,"Given the sequence, which activity had the lea...",open,"throwing a ball, skipping rope, shaking hands,..."
2684979,4.2.2,UMT5 Small,full,29995,4,1,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The sequence starts with throwing a ball. This...,right_before_open,What occurred right before someone was holding...,open,Someone is shaking hands.
2684980,4.2.2,UMT5 Small,full,29996,0,1,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The first activity the person is doing is danc...,last_open,What was the final action undertaken by the pe...,open,Someone is kicking a ball.
2684987,4.2.2,UMT5 Small,full,29997,2,1,"{'start': [0.0, 4.0, 8.0, 12.0], 'end': [4.0, ...",The first activity the person is doing is play...,extremum_least_open,Which activity had the smallest number of inst...,open,kicking a ball and eating with the right hand


In [13]:
# open.dropna().sample(200).to_hdf(base / "llmjudge_eval_set.h5", key="data")